# Customer Churn Prediction — Telco Dataset

**Dataset:** IBM Telco Customer Churn (7,043 customers · 21 features)  
**Goal:** Build and compare ML models to predict churn, with SHAP explainability and business insights  
**Author:** Berke Kutay

---
**Pipeline:** Data Loading → EDA → Preprocessing → SMOTE → Model Comparison → Tuning → SHAP → Business Insights

---
## 0. Environment Setup *(Colab only)*
Run cells 1–2 only if you're on Google Colab.

In [ ]:
# Run in Colab to upload the CSV
# from google.colab import files
# uploaded = files.upload()  # select: WA_Fn-UseC_-Telco-Customer-Churn.csv

In [ ]:
# Uncomment to install missing packages (Colab)
# !pip install -q optuna shap imbalanced-learn lightgbm xgboost
# print('Done.')

---
## 1. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, confusion_matrix, classification_report
)

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Global style ──────────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    'figure.figsize'   : (10, 6),
    'font.size'        : 12,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
})
sns.set_style('whitegrid')
COLORS = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']

print('Libraries loaded.')

---
## 2. Data Loading & Overview

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
# TotalCharges has blank strings for new customers (tenure=0) — coerce to NaN then fill
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'Missing TotalCharges: {df["TotalCharges"].isna().sum()} rows (tenure=0 customers)')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

df.describe()

In [ ]:
# Missing value check
missing = df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print('No missing values.')
else:
    fig, ax = plt.subplots(figsize=(14, 4))
    sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='Reds', ax=ax)
    ax.set_title('Missing Value Map')
    plt.tight_layout(); plt.show()

---
## 3. Exploratory Data Analysis

In [ ]:
# Churn distribution
churn_counts = df['Churn'].value_counts()
churn_pct    = churn_counts / len(df) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].pie(
    churn_counts, labels=['No Churn', 'Churn'],
    autopct='%1.1f%%', colors=COLORS[:2],
    startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Churn Distribution')

bars = axes[1].bar(['No Churn', 'Churn'], churn_counts, color=COLORS[:2], edgecolor='white')
for bar, n, p in zip(bars, churn_counts, churn_pct):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{n:,}\n({p:.1f}%)', ha='center', fontweight='bold')
axes[1].set_ylabel('Customers')
axes[1].set_ylim(0, churn_counts.max() * 1.18)
axes[1].set_title('Churn Count')

plt.suptitle('Target Variable Overview', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Numeric features by churn
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    sns.violinplot(data=df, x='Churn', y=col, palette=COLORS[:2], inner='box', ax=ax)
    ax.set_title(f'{col} by Churn')
plt.suptitle('Numeric Feature Distributions by Churn', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Contract type vs churn
contract_churn     = df.groupby(['Contract', 'Churn']).size().unstack(fill_value=0)
contract_churn_pct = contract_churn.div(contract_churn.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title in zip(
    axes,
    [contract_churn, contract_churn_pct],
    ['Count', '% of Contract Type']
):
    data.plot(kind='bar', ax=ax, color=COLORS[:2], edgecolor='white', rot=15)
    ax.set_title(f'Contract vs Churn ({title})')
    ax.set_xlabel(''); ax.legend(['No Churn', 'Churn'])
plt.tight_layout(); plt.show()

In [ ]:
# Churn rate by categorical features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat in zip(axes, ['InternetService', 'PaymentMethod', 'Contract']):
    rate = (df.groupby(feat)['Churn']
              .apply(lambda x: (x == 'Yes').mean() * 100)
              .sort_values(ascending=False))
    bars = ax.bar(rate.index, rate.values, color=COLORS[0], edgecolor='white')
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.5,
                f'{b.get_height():.1f}%', ha='center', fontsize=10)
    ax.set_title(f'Churn Rate by {feat}')
    ax.set_ylabel('Churn Rate (%)'); ax.tick_params(axis='x', rotation=20)
plt.suptitle('Churn Rate by Categorical Features', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap (numeric + churn binary)
num_df = df[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
num_df['Churn'] = (df['Churn'] == 'Yes').astype(int)

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(num_df.corr(), dtype=bool))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, square=True, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap (Numeric Features)')
plt.tight_layout(); plt.show()

---
## 4. Feature Engineering & Preprocessing

In [ ]:
df_model = df.copy()
df_model.drop(columns=['customerID'], inplace=True)

# Tenure group — ordinal feature capturing retention lifecycle
def tenure_label(t):
    if   t <= 12: return '0-12'
    elif t <= 24: return '12-24'
    elif t <= 48: return '24-48'
    elif t <= 60: return '48-60'
    else:         return '60+'

df_model['tenure_group'] = df_model['tenure'].apply(tenure_label)

# Binary encoding: Yes/No + gender
binary_map = {
    'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0,
    'No phone service': 0, 'No internet service': 0
}
binary_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn',
    'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]
for col in binary_cols:
    df_model[col] = df_model[col].map(binary_map)

# One-hot encoding for nominal columns
ohe_cols = ['PaymentMethod', 'Contract', 'InternetService', 'tenure_group']
df_model  = pd.get_dummies(df_model, columns=ohe_cols, drop_first=False)

# Boolean OHE columns → int (avoids sklearn warnings)
bool_cols = df_model.select_dtypes(include='bool').columns
df_model[bool_cols] = df_model[bool_cols].astype(int)

print(f'Final feature matrix: {df_model.shape}')
df_model.head()

In [ ]:
X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Churn rate — Train: {y_train.mean():.3f}  |  Test: {y_test.mean():.3f}')

---
## 5. Class Imbalance — SMOTE

In [ ]:
print('Before SMOTE:', dict(y_train.value_counts()))
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print('After  SMOTE:', dict(pd.Series(y_train_sm).value_counts()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (title, series) in zip(axes, [
    ('Before SMOTE', y_train),
    ('After SMOTE',  pd.Series(y_train_sm))
]):
    counts = series.value_counts()
    bars = ax.bar(['No Churn', 'Churn'], counts, color=COLORS[:2], edgecolor='white')
    for b, c in zip(bars, counts):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 30,
                f'{c:,}', ha='center', fontweight='bold')
    ax.set_title(title); ax.set_ylabel('Count')
plt.suptitle('Class Distribution: Before vs After SMOTE', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 6. Model Training & Comparison

Four models trained with identical pipelines:
- **StandardScaler** on numeric features only (`tenure`, `MonthlyCharges`, `TotalCharges`)
- **SMOTE** applied *inside* each CV fold — no data leakage into validation sets
- **5-fold stratified CV** on original training set → honest AUC estimate
- Final metrics evaluated on the original held-out test set

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline

NUMERIC_FEATS = ['tenure', 'MonthlyCharges', 'TotalCharges']

preprocessor = ColumnTransformer(
    [('scaler', StandardScaler(), NUMERIC_FEATS)],
    remainder='passthrough'
)

model_defs = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest'      : RandomForestClassifier(
                               n_estimators=200, max_depth=15,
                               min_samples_leaf=5,
                               random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost'            : XGBClassifier(n_estimators=200, eval_metric='logloss',
                                          verbosity=0, random_state=RANDOM_STATE, n_jobs=-1),
    'LightGBM'           : LGBMClassifier(n_estimators=200, verbose=-1,
                                           random_state=RANDOM_STATE, n_jobs=-1),
}

cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

for name, clf in model_defs.items():
    # SMOTE pipeline içinde — her CV fold'da ayrı uygulanır (leakage yok)
    pipe = ImbPipeline([
        ('pre',   preprocessor),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('clf',   clf)
    ])

    cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv,
                              scoring='roc_auc', n_jobs=-1)
    pipe.fit(X_train, y_train)
    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    results[name] = {
        'pipe'     : pipe,
        'y_pred'   : y_pred,
        'y_proba'  : y_proba,
        'cv_auc_mu': cv_auc.mean(),
        'cv_auc_sd': cv_auc.std(),
        'Accuracy' : accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall'   : recall_score(y_test, y_pred),
        'F1'       : f1_score(y_test, y_pred),
        'AUC-ROC'  : roc_auc_score(y_test, y_proba),
        'AP'       : average_precision_score(y_test, y_proba),
    }
    print(f'{name:22s}  CV AUC {cv_auc.mean():.4f}±{cv_auc.std():.4f}  '
          f'Test AUC {results[name]["AUC-ROC"]:.4f}  F1 {results[name]["F1"]:.4f}')

print('\nAll models trained.')

In [ ]:
# Overfitting kontrolü — CV AUC vs Test AUC (asıl metrik)
print(f"{'Model':<22} {'Train AUC':>10} {'CV AUC':>10} {'Test AUC':>10} {'CV-Test':>8}  Durum")
print("-" * 75)
for name, res in results.items():
    train_auc = roc_auc_score(y_train, res['pipe'].predict_proba(X_train)[:, 1])
    test_auc  = res['AUC-ROC']
    cv_auc    = res['cv_auc_mu']
    cv_test   = abs(cv_auc - test_auc)
    # Asıl kriter: CV ≈ Test (SMOTE leakage giderildikten sonra)
    flag = "✅ OK" if cv_test < 0.02 else "⚠️  CV-Test farkı büyük"
    print(f"{name:<22} {train_auc:>10.4f} {cv_auc:>10.4f} {test_auc:>10.4f} {cv_test:>8.4f}  {flag}")

print()
print("ℹ️  Tree modellerde yüksek Train AUC normaldir.")
print("   Önemli olan CV AUC ≈ Test AUC — model görünmeyen veriye iyi genelliyorsa sorun yok.")

---
## 7. Model Evaluation

In [ ]:
# Metrics table
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
comparison = pd.DataFrame(
    {n: {m: results[n][m] for m in metrics} for n in results}
).T.round(4)

display(comparison.style
    .highlight_max(axis=0, color='#c8e6c9')
    .format('{:.4f}')
    .set_caption('Model Comparison — Test Set')
)

In [ ]:
# ROC curves + Precision-Recall curves side by side
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for (name, res), color in zip(results.items(), COLORS):
    # ROC
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    axes[0].plot(fpr, tpr, color=color, lw=2,
                 label=f"{name} ({res['AUC-ROC']:.3f})")
    # PR
    prec, rec, _ = precision_recall_curve(y_test, res['y_proba'])
    axes[1].plot(rec, prec, color=color, lw=2,
                 label=f"{name} (AP={res['AP']:.3f})")

# ROC
axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves'); axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(alpha=0.3)

# PR
baseline = y_test.mean()
axes[1].axhline(baseline, color='k', linestyle='--', lw=1, label=f'Baseline ({baseline:.2f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves'); axes[1].legend(loc='upper right', fontsize=10)
axes[1].grid(alpha=0.3)

plt.suptitle('ROC & Precision-Recall Curves — All Models', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Best model: confusion matrix + classification report
best_name = comparison['AUC-ROC'].idxmax()
best_res  = results[best_name]
print(f'Best model: {best_name}  (AUC={best_res["AUC-ROC"]:.4f}  F1={best_res["F1"]:.4f})')

cm = confusion_matrix(y_test, best_res['y_pred'])
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'], linewidths=0.5)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout(); plt.show()

print(classification_report(y_test, best_res['y_pred'],
                             target_names=['No Churn', 'Churn']))

---
## 8. Hyperparameter Tuning — Optuna

Bayesian optimization (TPE sampler) over 40 trials on XGBoost.  
CV uses 3 folds here to keep tuning fast; final evaluation is on the held-out test set.

In [ ]:
# Fit scaler on original train set; reuse for Optuna and SHAP
scaler = StandardScaler()
X_train_sc = X_train.copy()
X_test_sc  = X_test.copy()
X_train_sc[NUMERIC_FEATS] = scaler.fit_transform(X_train[NUMERIC_FEATS])
X_test_sc[NUMERIC_FEATS]  = scaler.transform(X_test[NUMERIC_FEATS])

def objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 500),
        'max_depth'       : trial.suggest_int('max_depth', 3, 10),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'eval_metric'     : 'logloss',
        'verbosity'       : 0,
        'n_jobs'          : -1,
        'random_state'    : RANDOM_STATE,
    }
    # SMOTE her CV fold içinde uygulanır
    pipe = ImbPipeline([
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('clf',   XGBClassifier(**params))
    ])
    score = cross_val_score(pipe, X_train_sc, y_train,
                             cv=3, scoring='roc_auc', n_jobs=-1).mean()
    return score

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study.optimize(objective, n_trials=40, show_progress_bar=True)

print(f'\nBest CV AUC : {study.best_value:.4f}')
print(f'Best params : {study.best_params}')

In [ ]:
# Optuna — parameter importance plot
ax = optuna.visualization.matplotlib.plot_param_importances(study)
ax.figure.set_size_inches(9, 5)
ax.set_title('Optuna: Hyperparameter Importance', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Train tuned XGBoost — SMOTE pipeline ile, orijinal train seti üzerinde
tuned_xgb_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf',   XGBClassifier(
        **study.best_params,
        eval_metric='logloss', verbosity=0, n_jobs=-1
    ))
])
tuned_xgb_pipe.fit(X_train_sc, y_train)
tuned_xgb = tuned_xgb_pipe.named_steps['clf']  # SHAP için direkt modele erişim

y_pred_tuned  = tuned_xgb_pipe.predict(X_test_sc)
y_proba_tuned = tuned_xgb_pipe.predict_proba(X_test_sc)[:, 1]

# Before vs after comparison
xgb_base = results['XGBoost']
tune_df = pd.DataFrame({
    'XGBoost (baseline)': {
        'AUC-ROC': xgb_base['AUC-ROC'], 'F1': xgb_base['F1'],
        'Recall' : xgb_base['Recall'],  'Precision': xgb_base['Precision']
    },
    'XGBoost (tuned)': {
        'AUC-ROC': roc_auc_score(y_test, y_proba_tuned),
        'F1'     : f1_score(y_test, y_pred_tuned),
        'Recall' : recall_score(y_test, y_pred_tuned),
        'Precision': precision_score(y_test, y_pred_tuned),
    }
}).T

display(tune_df.round(4).style
    .highlight_max(axis=0, color='#c8e6c9')
    .format('{:.4f}')
    .set_caption('Tuning Impact — XGBoost')
)

---
## 9. Native Feature Importance

In [ ]:
# XGBoost built-in feature importances (gain metric)
feat_imp = pd.Series(
    tuned_xgb.feature_importances_,
    index=X_test.columns
).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.Blues_r(np.linspace(0.2, 0.8, len(feat_imp)))
bars = ax.barh(feat_imp.index[::-1], feat_imp.values[::-1], color=colors[::-1])
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('Top 20 Features — XGBoost Native Importance')
plt.tight_layout(); plt.show()

---
## 10. Classification Threshold Optimization

In [ ]:
# F1 / Precision / Recall vs threshold
thresholds = np.linspace(0.01, 0.99, 200)
f1s, precs, recs = [], [], []

for t in thresholds:
    pred = (y_proba_tuned >= t).astype(int)
    f1s.append(f1_score(y_test, pred, zero_division=0))
    precs.append(precision_score(y_test, pred, zero_division=0))
    recs.append(recall_score(y_test, pred, zero_division=0))

best_t = thresholds[np.argmax(f1s)]
print(f'Optimal threshold (max F1): {best_t:.3f}')
print(f'  F1        : {max(f1s):.4f}')
print(f'  Precision : {precs[np.argmax(f1s)]:.4f}')
print(f'  Recall    : {recs[np.argmax(f1s)]:.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1s,   label='F1',        color=COLORS[0], lw=2)
ax.plot(thresholds, precs, label='Precision', color=COLORS[2], lw=2, linestyle='--')
ax.plot(thresholds, recs,  label='Recall',    color=COLORS[1], lw=2, linestyle=':')
ax.axvline(best_t, color='gray', linestyle='--', lw=1.5, label=f'Best threshold ({best_t:.2f})')
ax.axvline(0.5,    color='black', linestyle=':', lw=1,   label='Default (0.50)')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Score')
ax.set_title('F1 / Precision / Recall vs Classification Threshold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Final predictions using optimal threshold
y_pred_opt = (y_proba_tuned >= best_t).astype(int)
print(f'\nWith threshold {best_t:.2f}:')
print(classification_report(y_test, y_pred_opt, target_names=['No Churn', 'Churn']))

---
## 11. Model Explainability — SHAP

SHAP (SHapley Additive exPlanations) distributes each prediction's output fairly across all features, giving both global and per-customer explanations.

In [ ]:
explainer   = shap.TreeExplainer(tuned_xgb)
shap_values = explainer.shap_values(X_test_sc)
feature_names = list(X_test.columns)
print(f'SHAP values computed: {X_test_sc.shape[0]} samples × {len(feature_names)} features')

In [ ]:
# SHAP beeswarm — global overview
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_sc, feature_names=feature_names,
                  plot_type='dot', max_display=15, show=False)
plt.title('SHAP Summary — Feature Impact on Churn Probability', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# SHAP bar — mean absolute importance
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_sc, feature_names=feature_names,
                  plot_type='bar', max_display=15, show=False)
plt.title('SHAP Global Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Waterfall — most at-risk customer
high_risk_idx = int(np.argmax(y_proba_tuned))
print(f'Customer #{high_risk_idx}')
print(f'  Predicted churn probability : {y_proba_tuned[high_risk_idx]:.3f}')
print(f'  Actual label                : {"Churn" if y_test.iloc[high_risk_idx] == 1 else "No Churn"}')

exp_val = explainer.expected_value
shap_exp = shap.Explanation(
    values       = shap_values[high_risk_idx],
    base_values  = exp_val,
    data         = np.array(X_test_sc.iloc[high_risk_idx]),
    feature_names= feature_names
)
plt.figure(figsize=(12, 5))
shap.waterfall_plot(shap_exp, max_display=12, show=False)
plt.title(f'Local Explanation — Customer #{high_risk_idx} (Highest Churn Risk)',
          fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Top-5 SHAP features summary
mean_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_names).sort_values(ascending=False)
print('Top 5 Churn Drivers (mean |SHAP|):')
for i, (feat, val) in enumerate(mean_shap.head(5).items(), 1):
    print(f'  {i}. {feat:<40} {val:.4f}')

---
## 12. Business Insights

### High-risk customer profile

A customer has the highest churn probability when they are:
- On a **month-to-month** contract
- In their **first 12 months** of tenure
- Using **fiber optic** internet
- Without **TechSupport** or **OnlineSecurity** add-ons
- Paying by **electronic check**
- Monthly charges **above $65**

### Key patterns from EDA + SHAP

| Driver | Finding |
|--------|---------|
| Contract type | Month-to-month churn rate ~42% vs ~3% for 2-year |
| Tenure | Churn risk halves after month 12, nearly disappears after month 24 |
| Monthly charges | Each $10 increase adds meaningful churn probability |
| Internet service | Fiber optic customers churn 2× more than DSL customers |
| Payment method | Electronic check users churn ~45%; credit card/bank transfer ~15–18% |

### 3 Actionable Recommendations

**1. Early Retention Program (Months 1–6)**  
Assign automated check-ins and a loyalty discount offer at month 3 to all new customers on month-to-month plans. This targets the highest-risk window identified in the tenure analysis.

**2. Contract Upgrade Incentives**  
Proactively offer month-to-month customers a 10–15% discount to switch to 1-year contracts. The SHAP analysis shows contract type is the single strongest churn predictor — even small conversion rates here will meaningfully reduce overall churn.

**3. Fiber Premium Bundle**  
Fiber customers churn at 42% while paying the highest bills. Bundle TechSupport + OnlineSecurity into a "Fiber Premium" tier. SHAP shows these features are among the top protective factors — and the perceived value increase justifies a modest price uplift.

---
## 13. Conclusion

In [ ]:
# Final leaderboard
final = comparison.copy()
final.loc['XGBoost (Tuned, t=0.50)'] = {
    'Accuracy' : accuracy_score(y_test, y_pred_tuned),
    'Precision': precision_score(y_test, y_pred_tuned),
    'Recall'   : recall_score(y_test, y_pred_tuned),
    'F1'       : f1_score(y_test, y_pred_tuned),
    'AUC-ROC'  : roc_auc_score(y_test, y_proba_tuned),
}
final.loc[f'XGBoost (Tuned, t={best_t:.2f})'] = {
    'Accuracy' : accuracy_score(y_test, y_pred_opt),
    'Precision': precision_score(y_test, y_pred_opt),
    'Recall'   : recall_score(y_test, y_pred_opt),
    'F1'       : f1_score(y_test, y_pred_opt),
    'AUC-ROC'  : roc_auc_score(y_test, y_proba_tuned),
}
display(final.round(4).style
    .highlight_max(axis=0, color='#c8e6c9')
    .format('{:.4f}')
    .set_caption('Final Model Leaderboard')
)

### Summary

- **Best model:** XGBoost tuned with Optuna — highest AUC-ROC and F1 across all configurations
- **Threshold tuning** further improves recall, which matters more in churn (missing a churner is costlier than a false alarm)
- **Top churn drivers:** contract type → tenure → monthly charges → fiber optic → lack of support services

### Limitations & Next Steps

- Dataset is a single snapshot — no time-series signals (usage trends, support tickets)
- SMOTE generates synthetic samples; real minority class behavior may differ
- Next steps: deploy as a REST API scoring service, A/B test retention campaigns on model-identified segments, explore survival analysis for time-to-churn estimation